# Forest Fires in Greece: Municipality Map

Map the Greek fire-service incident records by municipality. Circle markers show municipality-level reported burned area; the points are representative centers of municipal boundaries, **not exact fire coordinates**.

The source CSV is `data/mcp_forest_fires-f2bzgog3/export_mcp_forest_fires.csv`. Administrative boundaries are saved in `data/geoBoundaries-GRC-ADM3-simplified.geojson`, with source and license details in `data/SOURCES.md`.

Greek municipality names are preserved. The boundary source uses Latin-script names, so an editable crosswalk is created at `data/municipality_boundary_crosswalk.csv`. High-confidence suggestions are mapped; uncertain names remain unlocated until reviewed.

## Part 1. Load incidents and calculate area

The CSV already includes per-incident burned area by land category. This cell keeps the original Greek place names and calculates both all-category burned area and the forest, woodland, and grove subtotal.

In [1]:
from pathlib import Path

import pandas as pd

fires_path = Path("../data/mcp_forest_fires-f2bzgog3/export_mcp_forest_fires.csv")
fires = pd.read_csv(fires_path, parse_dates=["start_time", "end_time"])
area_columns = [column for column in fires.columns if column.endswith("_area_burned")]
fires[area_columns] = fires[area_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
fires["total_burned_ha"] = fires[area_columns].sum(axis=1)
fires["forest_burned_ha"] = fires[["forest_area_burned", "woodland_area_burned", "grove_area_burned"]].sum(axis=1)

print("fire incidents:", len(fires))
print("date range:", fires["start_time"].min().date(), "to", fires["start_time"].max().date())
print("municipalities:", fires["municipality"].nunique())
print("total reported burned area (ha):", round(fires["total_burned_ha"].sum(), 2))
print("forest, woodland, and grove area (ha):", round(fires["forest_burned_ha"].sum(), 2))
display(fires[["start_time", "prefecture", "municipality", "address", "total_burned_ha", "forest_burned_ha"]].head())

fire incidents: 43571
date range: 2014-01-02 to 2018-12-31
municipalities: 303
total reported burned area (ha): 1209195.18
forest, woodland, and grove area (ha): 573987.66


,start_time,prefecture,municipality,address,total_burned_ha,forest_burned_ha
0,2014-01-02 11:33:00,ΞΑΝΘΗΣ,Δ. ΑΒΔΗΡΩΝ,ΑΓΡΟΤΙΚΗ ΠΕΡΙΟΧΗ ΚΕΣΣΑΝΗΣ,1.0,0.0
1,2014-01-02 17:55:00,ΡΟΔΟΠΗΣ,Δ. ΚΟΜΟΤΗΝΗΣ,ΙΤΕΑ,1.0,0.0
2,2014-01-03 06:24:00,ΡΟΔΟΠΗΣ,Δ. ΜΑΡΩΝΕΙΑΣ - ΣΑΠΩΝ,ΟΙΚΙΣΜΟΣ ΠΟΝΤΙΩΝ,0.1,0.0
3,2014-01-03 17:40:00,ΑΧΑΙΑΣ,Δ. ΠΑΤΡΕΩΝ,Δ.Δ. ΒΕΛΒΙΤΣΙΟΥ,0.1,0.0
4,2014-01-03 18:54:00,ΛΕΣΒΟΥ,Δ. ΛΗΜΝΟΥ,ΚΟΤΣΙΝΑΣ,0.5,0.0


## Part 2. Match Greek municipalities to boundary names

The incident file has Greek municipality names, while the saved GeoJSON uses Latin-script names. This cell creates a reviewable crosswalk. Exact normalized matches are pre-approved; fuzzy suggestions are shown for you to review but are not mapped automatically. Edit `approved_boundary_name` in the CSV to confirm or correct a match.

In [2]:
import difflib
import json
import re
import unicodedata


boundaries_path = Path("../data/geoBoundaries-GRC-ADM3-simplified.geojson")
crosswalk_path = Path("../data/municipality_boundary_crosswalk.csv")
with boundaries_path.open(encoding="utf-8") as file:
    boundary_geojson = json.load(file)


def normalize_place_name(value):
    text = unicodedata.normalize("NFD", str(value).upper())
    text = "".join(char for char in text if unicodedata.category(char) != "Mn")
    digraphs = [("ΟΥ", "OU"), ("ΑΙ", "AI"), ("ΕΙ", "EI"), ("ΟΙ", "OI"), ("ΥΙ", "YI"), ("ΑΥ", "AV"), ("ΕΥ", "EV"), ("ΜΠ", "MP"), ("ΝΤ", "NT"), ("ΓΓ", "NG"), ("ΓΚ", "GK"), ("ΤΣ", "TS"), ("ΤΖ", "TZ")]
    for greek, latin in digraphs:
        text = text.replace(greek, latin)
    greek_letters = {"Α": "A", "Β": "V", "Γ": "G", "Δ": "D", "Ε": "E", "Ζ": "Z", "Η": "I", "Θ": "TH", "Ι": "I", "Κ": "K", "Λ": "L", "Μ": "M", "Ν": "N", "Ξ": "X", "Ο": "O", "Π": "P", "Ρ": "R", "Σ": "S", "Τ": "T", "Υ": "Y", "Φ": "F", "Χ": "CH", "Ψ": "PS", "Ω": "O"}
    text = text.translate(str.maketrans(greek_letters))
    return re.sub(r"[^A-Z0-9]", "", text)


boundary_names = [feature["properties"]["shapeName"] for feature in boundary_geojson["features"]]
boundary_by_key = {normalize_place_name(name): name for name in boundary_names}
municipalities = sorted(fires["municipality"].dropna().unique())

suggestions = []
for municipality in municipalities:
    greek_name = re.sub(r"^\s*Δ\.\s*", "", municipality).strip()
    key = normalize_place_name(greek_name)
    exact_name = boundary_by_key.get(key)
    close_key = difflib.get_close_matches(key, boundary_by_key.keys(), n=1, cutoff=0.5)
    suggested_name = exact_name or (boundary_by_key[close_key[0]] if close_key else "")
    score = 1.0 if exact_name else (difflib.SequenceMatcher(None, key, close_key[0]).ratio() if close_key else 0.0)
    suggestions.append({"municipality_greek": municipality, "municipality_label": greek_name, "suggested_boundary_name": suggested_name, "match_score": round(score, 3), "approved_boundary_name": exact_name or ""})

suggestions_df = pd.DataFrame(suggestions)
if crosswalk_path.exists():
    reviewed = pd.read_csv(crosswalk_path, keep_default_na=False).set_index("municipality_greek")
    reviewed_names = set(reviewed.index)
    for index, row in suggestions_df.iterrows():
        if row["municipality_greek"] in reviewed_names:
            suggestions_df.at[index, "approved_boundary_name"] = reviewed.at[row["municipality_greek"], "approved_boundary_name"]

suggestions_df.to_csv(crosswalk_path, index=False, encoding="utf-8-sig")

municipality_totals = (
    fires.groupby("municipality", dropna=False)
    .agg(incident_count=("start_time", "size"), total_burned_ha=("total_burned_ha", "sum"), forest_burned_ha=("forest_burned_ha", "sum"))
    .reset_index()
    .merge(suggestions_df[["municipality_greek", "municipality_label", "suggested_boundary_name", "match_score", "approved_boundary_name"]], left_on="municipality", right_on="municipality_greek", how="left")
)
print("municipality names:", len(suggestions_df))
print("exact matches:", int(suggestions_df["match_score"].eq(1).sum()))
print("review file:", crosswalk_path)
display(suggestions_df.sort_values(["match_score", "municipality_greek"], ascending=[True, True]).head(12))

municipality names: 303
exact matches: 23
review file: ../data/municipality_boundary_crosswalk.csv


,municipality_greek,municipality_label,suggested_boundary_name,match_score,approved_boundary_name
204,Δ. ΝΟΤΙΟΥ ΠΗΛΙΟΥ,ΝΟΤΙΟΥ ΠΗΛΙΟΥ,Topirou,0.421,South Pelion
191,Δ. ΝΑΞΟΥ & ΜΙΚΡΩΝ ΚΥΚΛΑΔΩΝ,ΝΑΞΟΥ & ΜΙΚΡΩΝ ΚΥΚΛΑΔΩΝ,Farkadona,0.429,Naxos and Lesser Cyclades
271,Δ. ΣΥΡΟΥ-ΕΡΜΟΥΠΟΛΕΩΣ,ΣΥΡΟΥ-ΕΡΜΟΥΠΟΛΕΩΣ,Petroupoli,0.462,Syros
77,Δ. ΔΥΤΙΚΗΣ ΑΧΑΪΑΣ,ΔΥΤΙΚΗΣ ΑΧΑΪΑΣ,Oichalia,0.545,West Achaea
78,Δ. ΔΥΤΙΚΗΣ ΜΑΝΗΣ,ΔΥΤΙΚΗΣ ΜΑΝΗΣ,Kaisariani,0.545,West Mani
246,Δ. ΡΟΔΟΥ,ΡΟΔΟΥ,Rhodes,0.545,Rhodes
212,Δ. ΟΡΟΠΕΔΙΟΥ ΛΑΣΙΘΙΟΥ,ΟΡΟΠΕΔΙΟΥ ΛΑΣΙΘΙΟΥ,Oraiokastro,0.552,Lasithi Plateau
114,Δ. ΙΕΡΑΣ ΠΟΛΗΣ ΜΕΣΟΛΟΓΓΙΟΥ,ΙΕΡΑΣ ΠΟΛΗΣ ΜΕΣΟΛΟΓΓΙΟΥ,Missolonghi,0.562,Missolonghi
140,Δ. ΚΕΡΚΥΡΑΣ,ΚΕΡΚΥΡΑΣ,Skyros,0.571,Corfu
29,Δ. ΑΝΑΤΟΛΙΚΗΣ ΜΑΝΗΣ,ΑΝΑΤΟΛΙΚΗΣ ΜΑΝΗΣ,Santorini,0.583,East Mani


## Part 3. Draw the municipality map

Municipal polygon fills classify reported burned area into five quantile classes. The administrative boundaries are not fire perimeters or fire locations, and the source does not report official fire-severity ratings.

In [7]:
import copy

import folium
from branca.element import MacroElement, Template

approved = municipality_totals.loc[
    municipality_totals["approved_boundary_name"].fillna("").ne("")
].copy()
mapped = (
    approved.groupby("approved_boundary_name", as_index=False)
    .agg(incident_count=("incident_count", "sum"), total_burned_ha=("total_burned_ha", "sum"), forest_burned_ha=("forest_burned_ha", "sum"), municipality_label=("municipality_label", "first"))
)
area_by_boundary = mapped.set_index("approved_boundary_name")["total_burned_ha"].to_dict()
features_for_map = copy.deepcopy(boundary_geojson)
for feature in features_for_map["features"]:
    boundary_name = feature["properties"]["shapeName"]
    feature["properties"]["total_burned_ha"] = area_by_boundary.get(boundary_name)

palette = ["#ffffb2", "#fecc5c", "#fd8d3c", "#f03b20", "#bd0026"]
class_breaks = mapped["total_burned_ha"].quantile([0.2, 0.4, 0.6, 0.8]).tolist()
class_labels = [
    f"Very low (≤ {class_breaks[0]:,.2f} ha)",
    f"Low (> {class_breaks[0]:,.2f}–{class_breaks[1]:,.2f} ha)",
    f"Moderate (> {class_breaks[1]:,.2f}–{class_breaks[2]:,.2f} ha)",
    f"High (> {class_breaks[2]:,.2f}–{class_breaks[3]:,.2f} ha)",
    f"Very high (> {class_breaks[3]:,.2f} ha)",
]


def boundary_style(feature):
    value = feature["properties"].get("total_burned_ha")
    if value is None:
        return {"fillColor": "#dfe7e3", "color": "#60726b", "weight": 0.7, "fillOpacity": 0.22}
    color_index = sum(value > threshold for threshold in class_breaks)
    return {"fillColor": palette[color_index], "color": "#305047", "weight": 1.0, "fillOpacity": 0.8}

fire_map = folium.Map(location=[38.3, 23.0], zoom_start=6, tiles="OpenStreetMap")
folium.GeoJson(
    features_for_map,
    name="Municipal burned area",
    style_function=boundary_style,
    tooltip=folium.GeoJsonTooltip(fields=["shapeName", "total_burned_ha"], aliases=["Boundary:", "Total burned area (ha):"], localize=True),
).add_to(fire_map)

legend_rows = "".join(
    f'<div style="display: flex; align-items: center; margin-bottom: 4px;"><span style="display: inline-block; width: 16px; height: 12px; background: {color}; border: 1px solid #7a9d93; margin-right: 8px;"></span>{label}</div>'
    for color, label in zip(palette, class_labels)
)
legend_rows += '<div style="display: flex; align-items: center;"><span style="display: inline-block; width: 16px; height: 12px; background: #dfe7e3; border: 1px solid #7a9d93; margin-right: 8px;"></span>No mapped data</div>'

legend_html = f"""
<div style="position: fixed; bottom: 30px; right: 30px; z-index: 9999; background: rgba(255,255,255,0.94); border: 2px solid #dfe7e3; border-radius: 10px; padding: 10px 12px; font-size: 12px; color: #1d2d2a; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
  <div style="font-weight: bold; margin-bottom: 6px;">Reported burned area (ha)</div>
  {legend_rows}
</div>
"""
legend = MacroElement()
legend._template = Template(legend_html)
fire_map.get_root().add_child(legend)

folium.LayerControl().add_to(fire_map)
map_path = Path("../site/fire_municipality_map.html")
map_path.parent.mkdir(parents=True, exist_ok=True)
fire_map.save(map_path)
print("mapped municipalities:", len(mapped))
print("interactive map saved to:", map_path)


mapped municipalities: 296
interactive map saved to: ../site/fire_municipality_map.html


Mapped municipalities are shown as colored administrative polygons. Unmatched names and incidents with no municipality value remain in the overall totals but are not shown on the map.

## Part 4. Remaining unmatched records

The crosswalk maps 302 of the 303 distinct municipality names to 296 distinct boundary labels after aliases are combined. `Δ. ΑΝΩΓΕΙΩΝ` remains unmatched because the supplied ADM3 layer has no corresponding boundary. Another 727 incidents have no municipality value in the source data. Together, these 732 incidents (16,905.53 ha) are not represented on the map but remain included in the overall and yearly totals.

In [4]:
unmatched = municipality_totals.loc[
    municipality_totals["approved_boundary_name"].fillna("").eq("")
].copy()
matched_event_count = int(mapped["incident_count"].sum())
matched_area_ha = float(mapped["total_burned_ha"].sum())
print("mapped municipalities:", len(mapped))
print("mapped incidents:", f"{matched_event_count:,}")
print("mapped burned area (ha):", f"{matched_area_ha:,.2f}")
print("unmatched or missing municipalities:", len(unmatched))
print("unmatched incidents:", f"{int(unmatched['incident_count'].sum()):,}")
print("unmatched burned area (ha):", f"{unmatched['total_burned_ha'].sum():,.2f}")
display(unmatched[["municipality", "suggested_boundary_name", "match_score", "incident_count", "total_burned_ha"]].sort_values("municipality"))

mapped municipalities: 296
mapped incidents: 42,839
mapped burned area (ha): 1,192,289.65
unmatched or missing municipalities: 2
unmatched incidents: 732
unmatched burned area (ha): 16,905.53


,municipality,suggested_boundary_name,match_score,incident_count,total_burned_ha
35,Δ. ΑΝΩΓΕΙΩΝ,Pangeou,0.667,5,6.70
303,NaN,NaN,NaN,727,16898.83


In [8]:
mapped_table = mapped.sort_values("total_burned_ha", ascending=False)
display(mapped_table[["municipality_label", "approved_boundary_name", "incident_count", "total_burned_ha", "forest_burned_ha"]])

,municipality_label,approved_boundary_name,incident_count,total_burned_ha,forest_burned_ha
11,ΚΙΛΕΛΕΡ,Kileler,1068,100654.40,1237.00
12,ΚΙΛΚΙΣ,Kilkis,452,12342.26,941.25
17,ΟΡΕΣΤΙΑΔΑΣ,Orestiadas,415,7266.69,462.00
6,ΔΡΑΜΑΣ,Dramas,143,5209.49,4176.04
13,ΚΟΜΟΤΗΝΗΣ,Komotinis,594,3575.40,225.92
0,ΑΛΕΞΑΝΔΡΟΥΠΟΛΗΣ,Alexandroupolis,304,3364.79,1170.26
4,ΔΙΔΥΜΟΤΕΙΧΟΥ,Didymoteichou,220,3163.00,822.60
14,ΜΕΓΑΛΟΠΟΛΗΣ,Megalopolis,95,2854.29,1170.78
16,ΝΕΣΤΟΥ,Nestou,406,2846.29,101.83
7,ΙΑΣΜΟΥ,Iasmou,261,2821.19,421.20


## Part 5. Report area by land category

The source records area separately for forests, woodland, groves, low vegetation, wetlands, agriculture, crop residue, and dumping grounds. The all-category total is not the same as forested area.

In [9]:
land_cover_totals = (
    fires[area_columns]
    .sum()
    .rename_axis("source_field")
    .reset_index(name="reported_area_ha")
)
land_cover_totals["land_category"] = (
    land_cover_totals["source_field"]
    .str.replace("_area_burned", "", regex=False)
    .str.replace("_", " ", regex=False)
    .str.title()
)
land_cover_totals = land_cover_totals[["land_category", "reported_area_ha"]].sort_values("reported_area_ha", ascending=False)
print("all-category area (ha):", f"{fires['total_burned_ha'].sum():,.2f}")
print("forest, woodland, grove (ha):", f"{fires['forest_burned_ha'].sum():,.2f}")
display(land_cover_totals)

all-category area (ha): 1,209,195.18
forest, woodland, grove (ha): 573,987.66


,land_category,reported_area_ha
1,Woodland,417695.92
6,Crop Residue,234611.59
3,Low Vegetation,192924.70
5,Agricultural,160011.35
0,Forest,155608.75
4,Swamp,43989.01
7,Dumping Ground,3670.87
2,Grove,682.99


## Part 6. Validate totals and map coverage

Mapped and unmatched totals should reconcile to the full source data. After editing the crosswalk, rerun the notebook from Part 2 onward to refresh the HTML map and these checks.

In [10]:
all_area_ha = float(fires["total_burned_ha"].sum())
unmatched_area_ha = float(unmatched["total_burned_ha"].sum())
assert len(fires) == 43571, "Unexpected incident row count; check the source file."
assert int(municipality_totals["incident_count"].sum()) == len(fires), "Municipality counts do not reconcile to source rows."
assert math.isclose(matched_area_ha + unmatched_area_ha, all_area_ha, rel_tol=1e-9), "Mapped and unmatched areas do not add to the source total."
assert map_path.exists(), "Interactive HTML map was not written."
assert crosswalk_path.exists(), "Municipality review crosswalk was not written."
print("All checks passed")
print(f"Source total: {all_area_ha:,.2f} ha")
print(f"Mapped total: {matched_area_ha:,.2f} ha ({matched_area_ha / all_area_ha:.1%})")
print(f"Unmatched total: {unmatched_area_ha:,.2f} ha ({unmatched_area_ha / all_area_ha:.1%})")
print("Map:", map_path)
print("Crosswalk:", crosswalk_path)

All checks passed
Source total: 1,209,195.18 ha
Mapped total: 158,957.85 ha (13.1%)
Unmatched total: 1,050,237.33 ha (86.9%)
Map: ../site/fire_municipality_map.html
Crosswalk: ../data/municipality_boundary_crosswalk.csv


## Sources and limitations

Open `../site/fire_municipality_map.html` to view the interactive map. Sources, access date, licenses, and boundary vintages are documented in `../data/SOURCES.md`.

The fire dataset supplies reported area totals but no fire coordinates or perimeter polygons. Map circles are approximate municipality representative points, and polygon shading aggregates reports to the matched municipality. Unmatched locations remain in the CSV crosswalk and in the overall area total; they are not shown on the map until reviewed.